# 01. 데이터 점검

Olist 주문 데이터를 분석에 쓰기 전에, 믿고 쓸 수 있는 상태인지 확인합니다.

1. 주문 상태와 도착 시각이 서로 맞는지
2. 어느 기간의 데이터를 써야 하는지
3. 테이블끼리 어떻게 연결되고, 합칠 때 어떤 함정이 있는지
4. 결제 테이블의 분할 결제 구조

In [1]:
import pandas as pd

orders = pd.read_csv("data/olist_orders_dataset.csv")

print(orders.shape)   # (행 수, 열 수)
orders.head()         # 위에서 5줄 미리보기

(99441, 8)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [2]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [3]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [4]:
mask = (orders["order_status"] == "delivered") & (orders["order_delivered_customer_date"].isna())
orders[mask]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaN,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaN,2018-07-19 00:00:00


In [5]:
mask.sum()

np.int64(8)

In [6]:
mask2 = (orders["order_status"] != "delivered") & (orders["order_delivered_customer_date"].notna())
mask2.sum()

np.int64(6)

In [7]:
orders[mask2]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
2921,1950d777989f6a877539f53795b4c3c3,1bccb206de9f0f25adc6871a1bcf77b2,canceled,2018-02-19 19:48:52,2018-02-19 20:56:05,2018-02-20 19:57:13,2018-03-21 22:03:51,2018-03-09 00:00:00
8791,dabf2b0e35b423f94618bf965fcb7514,5cdec0bb8cbdf53ffc8fdc212cd247c6,canceled,2016-10-09 00:56:52,2016-10-09 13:36:58,2016-10-13 13:36:59,2016-10-16 14:36:59,2016-11-30 00:00:00
58266,770d331c84e5b214bd9dc70a10b829d0,6c57e6119369185e575b36712766b0ef,canceled,2016-10-07 14:52:30,2016-10-07 15:07:10,2016-10-11 15:07:11,2016-10-14 15:07:11,2016-11-29 00:00:00
59332,8beb59392e21af5eb9547ae1a9938d06,bf609b5741f71697f65ce3852c5d2623,canceled,2016-10-08 20:17:50,2016-10-09 14:34:30,2016-10-14 22:45:26,2016-10-19 18:47:43,2016-11-30 00:00:00
92636,65d1e226dfaeb8cdc42f665422522d14,70fc57eeae292675927697fe03ad3ff5,canceled,2016-10-03 21:01:41,2016-10-04 10:18:57,2016-10-25 12:14:28,2016-11-08 10:58:34,2016-11-25 00:00:00
94399,2c45c33d2f9cb8ff8b1c86cc28c11c30,de4caa97afa80c8eeac2ff4c8da5b72e,canceled,2016-10-09 15:39:56,2016-10-10 10:40:49,2016-10-14 10:40:50,2016-11-09 14:53:50,2016-12-08 00:00:00


## 데이터 점검: 주문 상태와 도착 시각
- 주문 99,441건, 이 중 배송 완료 96,478건
- 배송 완료인데 도착 시각이 없는 주문: 8건
- 취소인데 도착 시각이 있는 주문: 6건 (5건이 2016년 10월, 반품 또는 초기 기록 오류로 추정)
- → 배송 분석은 "배송 완료 + 도착 시각 있음" 주문만 사용

In [8]:
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
orders["order_purchase_timestamp"].min(), orders["order_purchase_timestamp"].max()

(Timestamp('2016-09-04 21:15:19'), Timestamp('2018-10-17 17:30:18'))

In [9]:
orders["order_purchase_timestamp"].dt.to_period("M").value_counts().sort_index()

order_purchase_timestamp
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M, Name: count, dtype: int64

In [10]:
recent = orders[orders["order_purchase_timestamp"] >= "2018-09-01"]
recent["order_status"].value_counts()

order_status
canceled    19
shipped      1
Name: count, dtype: int64

In [11]:
nov = orders[(orders["order_purchase_timestamp"] >= "2017-11-01") & (orders["order_purchase_timestamp"] < "2017-12-01")]
nov["order_purchase_timestamp"].dt.date.value_counts().sort_index()

order_purchase_timestamp
2017-11-01     111
2017-11-02     124
2017-11-03     143
2017-11-04     111
2017-11-05     144
2017-11-06     193
2017-11-07     160
2017-11-08     175
2017-11-09     191
2017-11-10     165
2017-11-11     159
2017-11-12     174
2017-11-13     205
2017-11-14     192
2017-11-15     186
2017-11-16     226
2017-11-17     197
2017-11-18     149
2017-11-19     158
2017-11-20     230
2017-11-21     228
2017-11-22     201
2017-11-23     283
2017-11-24    1176
2017-11-25     499
2017-11-26     391
2017-11-27     403
2017-11-28     380
2017-11-29     323
2017-11-30     267
Name: count, dtype: int64

In [12]:
aug = orders[(orders["order_purchase_timestamp"] >= "2018-08-01") & (orders["order_purchase_timestamp"] < "2018-09-01")]
aug["order_status"].value_counts()

order_status
delivered      6351
canceled         84
shipped          47
invoiced         23
unavailable       7
Name: count, dtype: int64

## 데이터 점검: 주문 기간

### 1. 데이터 기간
- 첫 주문: 2016-09-04 21:15:19
- 마지막 주문: 2018-10-17 17:30:18

### 2. 제외할 기간
**2016년**
- 9월 4건, 10월 324건, 11월 0건, 12월 1건
- 주문 수가 너무 적고 빠진 달도 있어서 믿고 쓰기 어려움 → 제외

**2018년 9월 이후**
- 8월 6,512건이던 주문이 9월 16건, 10월 4건으로 급감
- 9월 이후 주문 20건 중 배송 완료 0건, 취소 19건, 배송 중 1건
- 8월은 배송 완료 비율이 약 97.5%로 다른 달과 비슷해 정상
- 9~10월이 왜 비었는지는 데이터만으로 확인할 수 없음 (데이터 제공 방식의 문제로 추정)
- → 8월까지만 사용하고 9월 이후는 제외

### 3. 특이한 달: 2017년 11월
- 11월 주문 7,544건으로 앞뒤 달(10월 4,631건, 12월 5,673건)보다 많음
- 11월 24일 하루에 1,176건, 평소 하루 110-200건의 약 6-7배
- 블랙프라이데이 할인 행사 때문 → 이상한 데이터가 아니라 실제 현상이므로 그대로 사용

### 4. 결론
- 월별 추세 분석은 **2017년 1월 ~ 2018년 8월 (20개월)** 데이터를 사용

In [13]:
orders["order_approved_at"] = pd.to_datetime(orders["order_approved_at"])

In [14]:
orders["order_delivered_carrier_date"] = pd.to_datetime(orders["order_delivered_carrier_date"])

In [15]:
orders["order_delivered_customer_date"] = pd.to_datetime(orders["order_delivered_customer_date"])

In [16]:
orders["order_estimated_delivery_date"] = pd.to_datetime(orders["order_estimated_delivery_date"])

In [17]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


In [18]:
customers = pd.read_csv("data/olist_customers_dataset.csv")
print(customers.shape)
customers.head()

(99441, 5)


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [19]:
items = pd.read_csv("data/olist_order_items_dataset.csv")
print(items.shape)
items.head()

(112650, 7)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [20]:
payments = pd.read_csv("data/olist_order_payments_dataset.csv")
print(payments.shape)
payments.head()

(103886, 5)


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [21]:
reviews = pd.read_csv("data/olist_order_reviews_dataset.csv")
print(reviews.shape)
reviews.head()

(99224, 7)


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [22]:
print("customer_id 개수:", customers["customer_id"].nunique())
print("실제 고객 수:", customers["customer_unique_id"].nunique())

customer_id 개수: 99441
실제 고객 수: 96096


In [23]:
order_count = customers["customer_unique_id"].value_counts()   # 고객별 주문 횟수
print("두 번 이상 산 고객:", (order_count >= 2).sum())
print("재구매 고객 비율:", round((order_count >= 2).mean() * 100, 1), "%")

두 번 이상 산 고객: 2997
재구매 고객 비율: 3.1 %


## 데이터 점검: 테이블 구조

### 1. 테이블별 행 수
| 테이블 | 행 수 | 한 줄의 의미 |
|---|---|---|
| orders | 99,441 | 주문 한 건 |
| customers | 99,441 | 주문 한 건의 고객 정보 |
| items | 112,650 | 주문에 담긴 상품 하나 |
| payments | 103,886 | 결제 한 번 |
| reviews | 99,224 | 리뷰 하나 |

### 2. 테이블 관계와 주의할 점
- items: 한 주문에 상품이 여러 개 → 주문 금액은 상품 가격을 더해서 계산
- payments: 한 주문을 여러 번 나눠 결제 → 그냥 합치면 주문이 중복으로 세어짐
- reviews: 리뷰가 없는 주문도 있음 → 리뷰 분석은 리뷰가 있는 주문만 대상
- customers: customer_id는 주문마다 새로 붙는 번호, 실제 사람은 customer_unique_id

### 3. 재구매 현황
- 주문 99,441건을 실제 고객 96,096명이 함
- 두 번 이상 산 고객 2,997명, 전체의 3.1%
- → 고객 대부분이 한 번 사고 다시 오지 않음. 재구매가 이 쇼핑몰의 핵심 과제

In [24]:
print("결제 테이블에 있는 주문 수:", payments["order_id"].nunique())
print("결제 순서 최댓값:", payments["payment_sequential"].max())
print("2번 이상 나눠 낸 결제 줄:", (payments["payment_sequential"] >= 2).sum())

결제 테이블에 있는 주문 수: 99440
결제 순서 최댓값: 29
2번 이상 나눠 낸 결제 줄: 4526


In [25]:
payments[payments["payment_sequential"] == 29]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
39108,fa65dad1b0e818e3ccc5cb0e39231352,29,voucher,1,19.26


In [26]:
one = payments[payments["order_id"] == "fa65dad1b0e818e3ccc5cb0e39231352"]
print("결제 줄 수:", len(one))
print("결제 수단:", one["payment_type"].unique())
print("총 결제 금액:", one["payment_value"].sum())
one.sort_values("payment_sequential")

결제 줄 수: 29
결제 수단: <StringArray>
['voucher']
Length: 1, dtype: str
총 결제 금액: 457.99


,order_id,payment_sequential,payment_type,payment_installments,payment_value
14321,fa65dad1b0e818e3ccc5cb0e39231352,1,voucher,1,3.71
23074,fa65dad1b0e818e3ccc5cb0e39231352,2,voucher,1,8.51
65641,fa65dad1b0e818e3ccc5cb0e39231352,3,voucher,1,2.95
9985,fa65dad1b0e818e3ccc5cb0e39231352,4,voucher,1,29.16
28330,fa65dad1b0e818e3ccc5cb0e39231352,5,voucher,1,0.66
29648,fa65dad1b0e818e3ccc5cb0e39231352,6,voucher,1,5.02
82593,fa65dad1b0e818e3ccc5cb0e39231352,7,voucher,1,0.32
68853,fa65dad1b0e818e3ccc5cb0e39231352,8,voucher,1,26.02
17274,fa65dad1b0e818e3ccc5cb0e39231352,9,voucher,1,1.08
19565,fa65dad1b0e818e3ccc5cb0e39231352,10,voucher,1,12.86


## 데이터 점검: 결제 테이블

### 1. 결제가 주문보다 많은 이유
| 단계 | 수 | 늘어난 이유 |
|---|---|---|
| 실제 고객 | 96,096명 | |
| 주문 | 99,441건 | 재구매 (한 사람이 여러 번 주문), 3,345건 증가 |
| 결제 | 103,886줄 | 분할 결제 (한 주문을 여러 수단으로 나눠 결제), 4,445줄 증가 |

- customers 테이블은 사람 수가 아니라 주문 수만큼 있으므로, 결제를 customers와 비교하는 것은 주문과 비교하는 것과 같음
- 따라서 결제가 주문보다 많은 이유는 재구매가 아니라 분할 결제

### 2. 결제 테이블 확인 결과
- 결제 기록이 있는 주문: 99,440건 (결제 기록이 없는 주문 1건)
- 결제 순서 최댓값: 29 (한 주문을 29번에 나눠 결제한 경우 있음)
- 결제 순서가 2 이상인 줄: 4,526줄
- 주문마다 첫 결제 한 줄씩을 빼면 추가된 줄은 4,446줄이어야 하는데 80줄 정도 많음
- → 결제 순서가 1번부터 시작하지 않는 주문이 80건 정도 있음 (기록 누락 또는 결제 취소로 추정)

### 3. 사례: 29번 나눠 결제한 주문
- 주문 번호: fa65dad1b0e818e3ccc5cb0e39231352
- 29건 모두 바우처(상품권), 금액은 0.00에서 150.00까지 제각각
- 금액이 0.00인 결제도 2건 있음
- 결제 시각 열이 없어서 한 번에 결제했는지 여러 번에 걸쳐 했는지는 알 수 없음
- 한 번의 결제에서 상품권 여러 장을 적용하고, 장마다 한 줄씩 기록된 것으로 추정

### 4. 분석할 때 지킬 원칙
- 주문 금액은 같은 주문 번호의 payment_value를 **전부 더해서** 계산
- 결제 순서 번호는 항상 1부터 시작하지 않으므로, 순서 번호에 의존하지 않기
- 주문과 결제 테이블을 합칠 때는 결제를 주문 단위로 먼저 더한 뒤 합치기 (그냥 합치면 주문이 중복으로 세어짐)